In [18]:
import pandas as pd

# Load the CSV files
df1 = pd.read_csv('../data/interim/predig_test_modf.csv')
df2 = pd.read_csv('../data/interim/predig_train_modf.csv')
myDf = pd.read_csv ('../data/processed/dataset_80_20_normalized_tcr_with_cohort.csv')

#Fuse the two dataframes
fused_df = pd.concat([df1, df2], ignore_index=True)

#print the shape of the fused dataframe
print(fused_df.shape)


(13073, 17)


In [19]:
fused_df.head()

,Epitope,HLA_allele,Immunogenicity,netcleave,NOAH,TAP,mhcflurry_affinity,mhcflurry_affinity_percentile,mhcflurry_processing_score,mhcflurry_presentation_score,mw_peptide,hydroph_peptide,charge_peptide,stab_peptide,mw_tcr_contact,hydroph_tcr_contact,charge_tcr_contact
0,DFMIQGGDF,HLA-A*24:02,1,0.656692,1.274,2.452,1934.222846,1.664625,0.268036,0.110196,1029.1321,0.077778,-2.001123,5.844444,488.4977,-0.660000,-1.001569
1,SLSKGDCQTL,HLA-A*02:01,0,0.941645,-6.555,1.012,185.979931,1.073875,0.075734,0.333421,1051.1796,-0.350000,-0.063839,-7.030000,650.7044,-1.583333,-0.063839
2,HLARHRHLM,HLA-B*08:01,0,0.417799,-4.144,0.250,33.034173,0.050875,0.004720,0.643370,1170.4037,-0.811111,2.270705,51.688889,717.8318,-2.320000,2.179796
3,SSQVLQQSTY,HLA-A*01:01,1,NaN,-5.107,3.012,53.093635,0.090125,0.180294,0.688690,1140.2150,-0.690000,-0.002866,135.780000,674.7519,-0.083333,-0.002016
4,QTYTLSQCF,HLA-B*15:01,0,0.940364,-3.689,2.794,129.970243,0.737000,0.170920,0.491236,1090.2158,-0.155556,-0.064841,12.477778,550.6274,0.260000,-0.063990


In [20]:
myDf.head()

,Cancer_Type,gene,Peptide,wt_seq,pep_mut_start,mutant_best_alleles,mutant_rank,mutant_rank_netMHCpan,mutant_rank_PRIME,mut_Rank_Stab,...,GTEx_all_tissues_expression_mean,Sample_Tissue_expression_GTEx,TCGA_Cancer_expression,bestWTMatchType_I,seq_len,immunogenicity,foreignness_score,hydrophobicity_avg_diff,hydrophobicity_total_diff,mut_Thalf
0,Melanoma,CHD5,KSKKEFRAY,KSEKEFRAY,3.0,HLA-C12:03,2.947,3.6904,2.947,0.8,...,8.816368,0.261257,0.206277,0.0,9,0,0.483,-0.044,-0.4,1.6604
1,Melanoma,MYOCD,QSFSSTSAL,QSSSSTSAL,3.0,HLA-B07:02,1.779,1.8105,1.779,3.0,...,10.009427,0.119799,0.314964,0.0,9,0,0.655,0.400,3.6,0.3099
2,Melanoma,GSE1,QAPDPAYIY,RAPDPAYIY,1.0,HLA-C07:02,0.039,0.0507,0.039,1.8,...,20.009704,12.388021,0.000000,0.0,9,0,0.366,0.111,1.0,0.1162
3,Melanoma,SUPT16H,GSDAEEGDSA,GSDAEEGDSE,10.0,HLA-A01:01,4.563,8.1864,4.563,42.0,...,41.991003,14.701189,54.687772,0.0,10,0,0.669,0.530,5.3,0.1289
4,Melanoma,FGF7,WYLRINKRGKV,WYLRIDKRGKV,6.0,HLA-A24:02,22.955,0.5629,22.955,3.5,...,15.394221,1.563364,7.467768,0.0,11,0,0.379,0.000,0.0,0.2800


# Quiero comprobar de estos cuantos epitopos yo no tengo en mi dataset

In [21]:
import pandas as pd
import numpy as np

def fusionar_y_eliminar_duplicados_con_prioridad(myDF, fused_df):
    """
    Fusiona myDF con fused_df, elimina duplicados y da prioridad a filas inmunogénicas.
    
    Parámetros:
    -----------
    myDF : pandas.DataFrame
        DataFrame principal con columnas que incluyen 'Peptide' y 'mutant_best_alleles'
    
    fused_df : pandas.DataFrame
        DataFrame secundario con columnas que incluyen 'Epitope', 'HLA_allele' e 'Immunogenicity'
    
    Retorna:
    --------
    pandas.DataFrame
        Nuevo DataFrame fusionado sin duplicados por Peptide + HLA
    """
    
    print("=" * 70)
    print("INICIANDO FUSIÓN DE DATASETS CON PRIORIDAD A INMUNOGÉNICOS")
    print("=" * 70)
    
    # PASO 1: Preparar fused_df - renombrar columnas y normalizar
    print("\n1️⃣ PREPARANDO DATASET 'PredIG'...")
    
    fused_preparado = fused_df.copy()
    
    # Renombrar columnas según especificaciones
    fused_preparado = fused_preparado.rename(columns={
        'Epitope': 'Peptide',
        'HLA_allele': 'mutant_best_alleles',
        'Immunogenicity': 'immunogenicity'
    })
    
    # Normalizar el formato HLA (eliminar asteriscos)
    fused_preparado['mutant_best_alleles'] = fused_preparado['mutant_best_alleles'].astype(str).str.replace('*', '', regex=False)
    
    # Añadir columna Cancer_Type con valor "PredIG" para todas las filas
    fused_preparado['Cancer_Type'] = 'PredIG'
    
    print(f"   ✓ Columnas renombradas (Epitope→Peptide, HLA_allele→mutant_best_alleles)")
    print(f"   ✓ Formato HLA normalizado (sin *)")
    print(f"   ✓ Añadida columna Cancer_Type = 'PredIG' para todas las filas")
    print(f"   ✓ Filas en PredIG preparado: {len(fused_preparado)}")
    
    # PASO 2: Identificar columnas
    print("\n2️⃣ IDENTIFICANDO ESTRUCTURA DE COLUMNAS...")
    
    columnas_myDF = set(myDF.columns)
    columnas_fused = set(fused_preparado.columns)
    
    columnas_nuevas = columnas_fused - columnas_myDF
    columnas_faltantes = columnas_myDF - columnas_fused
    
    print(f"   ✓ Columnas en myDF original: {len(columnas_myDF)}")
    print(f"   ✓ Columnas en PredIG: {len(columnas_fused)}")
    print(f"   ✓ Columnas nuevas de PredIG: {len(columnas_nuevas)}")
    
    if columnas_nuevas:
        print(f"     → {', '.join(list(columnas_nuevas)[:5])}" + 
              (f" ... y {len(columnas_nuevas)-5} más" if len(columnas_nuevas) > 5 else ""))
    
    if columnas_faltantes:
        print(f"   ⚠️  Columnas de myDF que no están en PredIG: {len(columnas_faltantes)}")
        print(f"     → {', '.join(list(columnas_faltantes)[:5])}" + 
              (f" ... y {len(columnas_faltantes)-5} más" if len(columnas_faltantes) > 5 else ""))
    
    # PASO 3: Fusionar datasets
    print("\n3️⃣ FUSIONANDO DATASETS...")
    
    # Asegurarse de que myDF tiene columna 'immunogenicity' si no existe
    if 'immunogenicity' not in myDF.columns:
        myDF['immunogenicity'] = np.nan
    
    # Concatenar ambos datasets
    df_fusionado = pd.concat([myDF, fused_preparado], ignore_index=True)
    
    print(f"   ✓ Filas en myDF original: {len(myDF)}")
    print(f"   ✓ Filas añadidas de PredIG: {len(fused_preparado)}")
    print(f"   ✓ Total filas después de fusión: {len(df_fusionado)}")
    
    # PASO 4: Preparar para eliminar duplicados con prioridad
    print("\n4️⃣ PREPARANDO PARA ELIMINAR DUPLICADOS CON PRIORIDAD...")
    
    # Normalizar HLA en todo el dataset fusionado
    df_fusionado['mutant_best_alleles'] = df_fusionado['mutant_best_alleles'].astype(str).str.replace('*', '', regex=False)
    
    # Asegurar que immunogenicity sea numérico para la comparación
    df_fusionado['immunogenicity'] = pd.to_numeric(df_fusionado['immunogenicity'], errors='coerce')
    
    # Identificar duplicados
    duplicados_mask = df_fusionado.duplicated(subset=['Peptide', 'mutant_best_alleles'], keep=False)
    duplicados_df = df_fusionado[duplicados_mask]
    
    print(f"   ✓ Filas duplicadas encontradas: {len(duplicados_df)}")
    
    if len(duplicados_df) > 0:
        combinaciones_unicas = duplicados_df[['Peptide', 'mutant_best_alleles']].drop_duplicates()
        print(f"   ✓ Combinaciones Peptide+HLA duplicadas: {len(combinaciones_unicas)}")
        
        # Contar cuántos duplicados tienen immunogenicity = 1
        inmunogenicos = duplicados_df[duplicados_df['immunogenicity'] == 1]
        print(f"   ✓ Duplicados inmunogénicos (immunogenicity=1): {len(inmunogenicos)}")
    
    # PASO 5: Eliminar duplicados con prioridad a inmunogénicos
    print("\n5️⃣ ELIMINANDO DUPLICADOS (PRIORIDAD A INMUNOGÉNICOS)...")
    
    # Función para seleccionar la mejor fila dentro de cada grupo de duplicados
    def seleccionar_mejor_fila(grupo):
        """
        Selecciona la mejor fila dentro de un grupo de duplicados.
        Prioridad: 1) immunogenicity=1, 2) primera fila
        """
        # Buscar filas inmunogénicas
        inmunogenicas = grupo[grupo['immunogenicity'] == 1]
        
        if not inmunogenicas.empty:
            # Si hay inmunogénicas, tomar la primera
            return inmunogenicas.iloc[0]
        else:
            # Si no hay inmunogénicas, tomar la primera fila del grupo
            return grupo.iloc[0]
    
    # Agrupar por Peptide + HLA y aplicar la función de selección
    grupos = df_fusionado.groupby(['Peptide', 'mutant_best_alleles'])
    
    # Lista para almacenar las filas seleccionadas
    filas_seleccionadas = []
    
    for (peptide, hla), grupo in grupos:
        if len(grupo) > 1:
            # Si hay duplicados, aplicar selección con prioridad
            mejor_fila = seleccionar_mejor_fila(grupo)
            filas_seleccionadas.append(mejor_fila)
        else:
            # Si no hay duplicados, mantener la fila
            filas_seleccionadas.append(grupo.iloc[0])
    
    # Crear el DataFrame final
    df_final = pd.DataFrame(filas_seleccionadas).reset_index(drop=True)
    
    # PASO 6: Estadísticas finales
    print("\n6️⃣ ESTADÍSTICAS FINALES:")
    print("-" * 40)
    
    # Contar orígenes de los datos
    orig_myDF = df_final[df_final['Cancer_Type'] != 'PredIG'].shape[0]
    orig_predig = df_final[df_final['Cancer_Type'] == 'PredIG'].shape[0]
    
    print(f"   Total filas finales: {len(df_final)}")
    print(f"   - De myDF original: {orig_myDF}")
    print(f"   - De PredIG: {orig_predig}")
    
    # Estadísticas de inmunogenicidad
    inmunogenicos_final = df_final[df_final['immunogenicity'] == 1].shape[0]
    print(f"   Filas inmunogénicas (immunogenicity=1): {inmunogenicos_final}")
    
    # Verificar si se mantuvieron los inmunogénicos de duplicados
    if len(duplicados_df) > 0:
        duplicados_inmuno = duplicados_df[duplicados_df['immunogenicity'] == 1]
        mantenidos = 0
        
        for _, row in duplicados_inmuno.iterrows():
            peptide = row['Peptide']
            hla = row['mutant_best_alleles']
            # Verificar si esta combinación está en el final
            if ((df_final['Peptide'] == peptide) & 
                (df_final['mutant_best_alleles'] == hla)).any():
                mantenidos += 1
        
        print(f"   Duplicados inmunogénicos mantenidos: {mantenidos}/{len(duplicados_inmuno)}")
    
    # Mostrar algunos ejemplos de duplicados resueltos
    print("\n📊 EJEMPLOS DE DUPLICADOS RESUELTOS:")
    print("-" * 50)
    
    if len(duplicados_df) > 0:
        # Tomar las primeras 3 combinaciones duplicadas
        for (peptide, hla), grupo in list(grupos)[:3]:
            if len(grupo) > 1:
                seleccionada = df_final[
                    (df_final['Peptide'] == peptide) & 
                    (df_final['mutant_best_alleles'] == hla)
                ].iloc[0]
                
                print(f"\n✅ {peptide} + {hla}:")
                print(f"   Duplicados originales: {len(grupo)}")
                print(f"   Seleccionada - Cancer_Type: {seleccionada['Cancer_Type']}, " +
                      f"immunogenicity: {seleccionada['immunogenicity']}")
                
                # Mostrar valores alternativos
                otros_inmuno = grupo[grupo['immunogenicity'] == 1]
                if len(otros_inmuno) > 0:
                    print(f"   Había {len(otros_inmuno)} opción(es) inmunogénica(s)")
    
    print("\n" + "=" * 70)
    print("✅ FUSIÓN COMPLETADA EXITOSAMENTE")
    print("=" * 70)
    
    return df_final

# Función para mostrar duplicados encontrados (antes de eliminarlos)
def mostrar_analisis_duplicados(myDF, fused_df):
    """
    Muestra un análisis detallado de los duplicados antes de fusionar
    """
    
    print("🔍 ANÁLISIS DE DUPLICADOS ENTRE DATASETS")
    print("-" * 50)
    
    # Preparar fused_df igual que en la función principal
    fused_preparado = fused_df.copy().rename(columns={
        'Epitope': 'Peptide',
        'HLA_allele': 'mutant_best_alleles',
        'Immunogenicity': 'immunogenicity'
    })
    fused_preparado['mutant_best_alleles'] = fused_preparado['mutant_best_alleles'].str.replace('*', '', regex=False)
    fused_preparado['Cancer_Type'] = 'PredIG'
    
    # Asegurar columna immunogenicity en myDF
    if 'immunogenicity' not in myDF.columns:
        myDF_copy = myDF.copy()
        myDF_copy['immunogenicity'] = np.nan
    else:
        myDF_copy = myDF.copy()
    
    # Normalizar HLA en ambos
    myDF_copy['mutant_best_alleles'] = myDF_copy['mutant_best_alleles'].astype(str).str.replace('*', '', regex=False)
    
    # Combinar temporalmente solo para el análisis
    temp_combinado = pd.concat([myDF_copy, fused_preparado], ignore_index=True)
    temp_combinado['immunogenicity'] = pd.to_numeric(temp_combinado['immunogenicity'], errors='coerce')
    
    # Encontrar duplicados
    duplicados = temp_combinado[temp_combinado.duplicated(['Peptide', 'mutant_best_alleles'], keep=False)]
    
    if duplicados.empty:
        print("No se encontraron duplicados entre los datasets.")
        return
    
    print(f"Se encontraron {len(duplicados[['Peptide', 'mutant_best_alleles']].drop_duplicates())} " +
          f"combinaciones duplicadas.")
    
    # Agrupar y mostrar
    grupos = duplicados.groupby(['Peptide', 'mutant_best_alleles'])
    
    print("\n📋 DETALLE DE DUPLICADOS (primeros 5):")
    print("-" * 60)
    
    for (peptide, hla), grupo in list(grupos)[:5]:
        print(f"\n🧬 {peptide} + {hla}:")
        print(f"   Total ocurrencias: {len(grupo)}")
        
        # Separar por origen
        de_myDF = grupo[grupo['Cancer_Type'] != 'PredIG']
        de_predig = grupo[grupo['Cancer_Type'] == 'PredIG']
        
        if not de_myDF.empty:
            print(f"   - De myDF: {len(de_myDF)}")
            immunogenicity_myDF = de_myDF['immunogenicity'].unique()
            print(f"     immunogenicity: {immunogenicity_myDF}")
        
        if not de_predig.empty:
            print(f"   - De PredIG: {len(de_predig)}")
            immunogenicity_predig = de_predig['immunogenicity'].unique()
            print(f"     immunogenicity: {immunogenicity_predig}")
        
        # Ver si hay inmunogénicos
        inmunogenicos = grupo[grupo['immunogenicity'] == 1]
        if not inmunogenicos.empty:
            print(f"   ⭐ Hay {len(inmunogenicos)} opción(es) inmunogénica(s)")
    
    if len(grupos) > 5:
        print(f"\n... y {len(grupos) - 5} combinaciones duplicadas más")

dataset_final = fusionar_y_eliminar_duplicados_con_prioridad(myDf, fused_df)

display(dataset_final.head(10))

INICIANDO FUSIÓN DE DATASETS CON PRIORIDAD A INMUNOGÉNICOS

1️⃣ PREPARANDO DATASET 'PredIG'...
   ✓ Columnas renombradas (Epitope→Peptide, HLA_allele→mutant_best_alleles)
   ✓ Formato HLA normalizado (sin *)
   ✓ Añadida columna Cancer_Type = 'PredIG' para todas las filas
   ✓ Filas en PredIG preparado: 13073

2️⃣ IDENTIFICANDO ESTRUCTURA DE COLUMNAS...
   ✓ Columnas en myDF original: 28
   ✓ Columnas en PredIG: 18
   ✓ Columnas nuevas de PredIG: 14
     → mhcflurry_affinity_percentile, mw_peptide, charge_tcr_contact, mhcflurry_affinity, NOAH ... y 9 más
   ⚠️  Columnas de myDF que no están en PredIG: 24
     → mutant_rank, DAI_MixMHC, DAI_NetStab, mut_netchop_score_ct, hydrophobicity_avg_diff ... y 19 más

3️⃣ FUSIONANDO DATASETS...
   ✓ Filas en myDF original: 11188
   ✓ Filas añadidas de PredIG: 13073
   ✓ Total filas después de fusión: 24261

4️⃣ PREPARANDO PARA ELIMINAR DUPLICADOS CON PRIORIDAD...
   ✓ Filas duplicadas encontradas: 6330
   ✓ Combinaciones Peptide+HLA duplicadas: 3

,Cancer_Type,gene,Peptide,wt_seq,pep_mut_start,mutant_best_alleles,mutant_rank,mutant_rank_netMHCpan,mutant_rank_PRIME,mut_Rank_Stab,...,mhcflurry_affinity_percentile,mhcflurry_processing_score,mhcflurry_presentation_score,mw_peptide,hydroph_peptide,charge_peptide,stab_peptide,mw_tcr_contact,hydroph_tcr_contact,charge_tcr_contact
0,PredIG,NaN,AAAAAAAAVL,NaN,NaN,HLA-B07:02,NaN,NaN,NaN,NaN,...,0.784250,0.304435,0.522584,798.9376,2.240000,-0.002016,9.000000,472.5418,2.20,-0.002016
1,PredIG,NaN,AAAAAAAVL,NaN,NaN,HLA-B07:02,NaN,NaN,NaN,NaN,...,0.296375,0.662384,0.927066,727.8588,2.288889,-0.002016,8.888889,401.4630,2.28,-0.002016
2,Melanoma,ZC4H2,AAAAAAQQIQV,AAAAAAQQLQV,9.0,HLA-B07:02,6.429,26.7692,6.429,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Melanoma,MAGEB2,AAAAAGVSFTK,AAAAAGVSSTK,9.0,A1101,0.510,0.2724,0.510,0.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Melanoma,MAGEB2,AAAAAGVSFTK,AAAAAGVSSTK,9.0,HLA-A11:01,0.510,0.1258,0.510,0.07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,PredIG,NaN,AAAAAIFVI,NaN,NaN,HLA-B51:01,NaN,NaN,NaN,NaN,...,0.284625,0.433790,0.713430,846.0372,2.777778,-0.002016,8.888889,519.6414,3.02,-0.002016
6,PredIG,NaN,AAAACTTMK,NaN,NaN,HLA-A03:01,NaN,NaN,NaN,NaN,...,0.066625,0.087179,0.747829,867.0461,0.700000,0.935715,93.933333,525.6356,0.96,-0.063990
7,Melanoma,MAGEB2,AAAAGVSFTK,AAAAGVSSTK,8.0,HLA-A11:01,0.112,0.4272,0.112,0.40,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,PredIG,NaN,AAAAGWQTL,NaN,NaN,HLA-A02:01,NaN,NaN,NaN,NaN,...,2.952250,0.267123,0.076686,887.9907,0.611111,-0.002016,22.600000,561.5949,-0.74,-0.002016
9,PredIG,NaN,AAAALHMQR,NaN,NaN,HLA-A11:01,NaN,NaN,NaN,NaN,...,0.280250,0.037705,0.635534,968.1417,0.188889,1.088890,0.511111,598.7178,0.16,0.088893


In [24]:
#Imprimo la cantidad de inmunogenicos = 1 e immunogenics = 0
inmunogenicos_1 = dataset_final[dataset_final['immunogenicity'] == 1].shape[0]
inmunogenicos_0 = dataset_final[dataset_final['immunogenicity'] == 0].shape[0] 
print(f"Cantidad de inmunogénicos (immunogenicity=1): {inmunogenicos_1}")
print(f"Cantidad de no inmunogénicos (immunogenicity=0): {inmunogenicos_0}")

Cantidad de inmunogénicos (immunogenicity=1): 7120
Cantidad de no inmunogénicos (immunogenicity=0): 13838


In [22]:
dataset_final.to_csv('../data/interim/dataset_fusionado_final.csv', index=False)